# Advanced Python Exceptions — 30+ Worked Problems, Solutions & Tests

**A self-contained, executable workbook.** Adapted and expanded from your provided lesson on Python exception objects, the `BaseException → Exception → LookupError → IndexError` hierarchy, propagation and stack traces, the dangers of `except Exception`, and generator error handling. Additional material is explicitly an advanced extension: chaining, custom exception contracts, resource safety, retries, asynchronous cancellation, and resilient pipelines.

**How to use:** Run top to bottom in a Python **3.11+** Jupyter kernel. For each numbered problem, read the challenge first, attempt it, then run the worked solution and its tests. All intentionally raised failures are caught and tested, so **Run All should finish without an uncaught exception**. Only Python's standard library is needed for the *notebook itself*. File-writing demonstrations use temporary directories. No networking, external credentials, or real waiting.

**Best-practice principles:** Catch exceptions you can meaningfully handle; use `else` for success-only work and `finally` / context managers for cleanup; maintain traceback causality; never silently disguise programming errors; make exception contracts and failure tests explicit. `except BaseException` is reserved for cleanup followed immediately by re-raising, not suppressing termination signals.

In [1]:
import asyncio
import contextlib
import importlib
import io
import json
import logging
import os
import sys
import tempfile
import traceback
import unittest
import warnings
from collections import deque
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path

assert sys.version_info >= (3, 11), 'Use a Python 3.11+ kernel for ExceptionGroup and TaskGroup examples.'

def expect_error(expected_type, fn, *args, contains=None, **kwargs):
    """Run fn; assert that it raises the expected exception; return the instance."""
    try:
        fn(*args, **kwargs)
    except expected_type as exc:
        if contains is not None:
            assert contains in str(exc), (contains, str(exc))
        return exc
    except BaseException as exc:
        raise AssertionError(
            f'Expected {expected_type}, got {type(exc).__name__}: {exc}'
        ) from exc
    raise AssertionError(f'Expected {expected_type} but no exception was raised')

print('Python:', sys.version.split()[0], '| standard-library setup: OK')

Python: 3.13.7 | standard-library setup: OK


---

## Part I — Exception semantics, matching, and control flow

Foundation: the original lesson establishes that exceptions are *objects*, handlers match subclasses, and propagation continues until a matching handler handles the error.

### Problem 01 — Hierarchy-aware dispatch

**Challenge.** Build `first_matching_handler(error, handlers)`, where `handlers` is an ordered sequence of exception classes. Return the *first matching class*; return `None` if none matches. Prove that `IndexError` matches `LookupError` and `Exception`, but the order of `except` branches matters. Reject non-exception classes in the handler configuration.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [2]:
def first_matching_handler(error, handlers):
    if not isinstance(error, BaseException):
        raise TypeError('error must be an exception instance')
    for candidate in handlers:
        if not isinstance(candidate, type) or not issubclass(candidate, BaseException):
            raise TypeError('each handler must be an exception class')
        if isinstance(error, candidate):
            return candidate
    return None

**Examples and automated checks** — execute this cell to verify edge cases:

In [3]:
err = IndexError('missing')
assert first_matching_handler(err, [LookupError, IndexError]) is LookupError
assert first_matching_handler(err, [IndexError, LookupError]) is IndexError
assert first_matching_handler(err, [ValueError, Exception]) is Exception
assert first_matching_handler(err, [ValueError]) is None
assert issubclass(IndexError, LookupError)
assert issubclass(LookupError, Exception)
expect_error(TypeError, first_matching_handler, err, [str])
print('01: first-match semantics and hierarchy verified')

01: first-match semantics and hierarchy verified


**Why this works / trade-offs.** Python selects the first `except` branch whose type matches the raised object or its ancestors. Put more specific handlers before general handlers; `except Exception` first would make later handlers unreachable in practice.

### Problem 02 — Precise indexed access vs masked bugs

**Challenge.** Implement a sequence accessor accepting *only nonnegative integer indexes*, with a caller-supplied default for out-of-range access. It must **not** suppress wrong-type indexes, and must distinguish an absent index from a valid element whose value is `None`.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [4]:
_MISSING_02 = object()
def get_at(seq, index, default=_MISSING_02):
    if type(index) is not int or index < 0:  # bool is deliberately not a valid index
        raise TypeError('index must be a nonnegative int')
    try:
        return seq[index]
    except IndexError:
        if default is _MISSING_02:
            raise
        return default

**Examples and automated checks** — execute this cell to verify edge cases:

In [5]:
assert get_at([None, 'b'], 0, 'missing') is None
assert get_at([None, 'b'], 10, 'missing') == 'missing'
assert get_at([10], 0) == 10
assert isinstance(expect_error(IndexError, get_at, [10], 3), IndexError)
expect_error(TypeError, get_at, [10], '0')
expect_error(TypeError, get_at, [10], True)
expect_error(TypeError, get_at, [10], -1)
print('02: sentinel and narrow IndexError handling verified')

02: sentinel and narrow IndexError handling verified


**Why this works / trade-offs.** A unique sentinel distinguishes an omitted default from a legitimate `None`. The `except IndexError` covers only the indexing operation, not unrelated computation. Integer and negative-index policies are explicit, rather than relying on surprising Python list semantics.

### Problem 03 — `try`/`except`/`else`/`finally` event order

**Challenge.** Write a function that records control-flow events for successful parsing and malformed input. Record `else` only for success, and `finally` on either path. No exceptions should escape this demonstration.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [6]:
def parse_event_trace(text):
    events = ['enter']
    try:
        number = int(text)
    except ValueError:
        events.append('except')
        return events
    else:
        events.append('else')
        events.append(number)
        return events
    finally:
        events.append('finally')

**Examples and automated checks** — execute this cell to verify edge cases:

In [7]:
assert parse_event_trace('12') == ['enter', 'else', 12, 'finally']
assert parse_event_trace('oops') == ['enter', 'except', 'finally']
print('03: else runs on success; finally runs before function exit')

03: else runs on success; finally runs before function exit


**Why this works / trade-offs.** A return expression is computed before `finally` runs; here the returned `events` list is the same mutable object, so the appended `finally` event appears in the returned list. Avoid mutating return values in `finally` in real code: the exercise isolates subtle execution order.

### Problem 04 — Preserve a traceback with a bare re-raise

**Challenge.** Create an inner failure, catch it in a middle layer, annotate it with `add_note`, then re-raise it with `raise` (not `raise exc`). Verify the outer layer sees the *same exception object*, its note, and frames from inner and middle.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [8]:
def deepest_04():
    raise ValueError('invalid payload')

def middle_04():
    try:
        deepest_04()
    except ValueError as exc:
        exc.add_note('while processing batch 04')
        raise

def outer_04():
    try:
        middle_04()
    except ValueError as exc:
        frames = [frame.name for frame in traceback.extract_tb(exc.__traceback__)]
        return exc, frames
    raise AssertionError('expected an error')

**Examples and automated checks** — execute this cell to verify edge cases:

In [9]:
exc_04, frames_04 = outer_04()
assert str(exc_04) == 'invalid payload'
assert 'while processing batch 04' in exc_04.__notes__
assert 'deepest_04' in frames_04 and 'middle_04' in frames_04
print('04: traceback frames:', ' → '.join(frames_04))

04: traceback frames: outer_04 → middle_04 → deepest_04


**Why this works / trade-offs.** The raised instance propagates through call frames. Bare `raise` inside an active handler preserves the original error and traceback without introducing an artificial re-raise location. Notes are available in Python 3.11+; add useful context without altering exception identity.

### Problem 05 — Explicit cause vs implicit context vs suppressed display

**Challenge.** Implement three wrappers around a failing integer parser: (a) `raise Custom from exc`, (b) `raise Custom` inside an `except`, and (c) `raise Custom from None`. Inspect `__cause__`, `__context__`, and `__suppress_context__`.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [10]:
class ParseFailure05(ValueError):
    pass

def explicit_05(text):
    try:
        return int(text)
    except ValueError as exc:
        raise ParseFailure05('bad number') from exc

def implicit_05(text):
    try:
        return int(text)
    except ValueError:
        raise ParseFailure05('bad number')

def suppressed_05(text):
    try:
        return int(text)
    except ValueError:
        raise ParseFailure05('public error') from None

**Examples and automated checks** — execute this cell to verify edge cases:

In [11]:
a05 = expect_error(ParseFailure05, explicit_05, 'n/a')
b05 = expect_error(ParseFailure05, implicit_05, 'n/a')
c05 = expect_error(ParseFailure05, suppressed_05, 'n/a')
assert isinstance(a05.__cause__, ValueError) and a05.__suppress_context__
assert a05.__context__ is a05.__cause__
assert b05.__cause__ is None and isinstance(b05.__context__, ValueError)
assert c05.__cause__ is None and isinstance(c05.__context__, ValueError)
assert c05.__suppress_context__
print('05: explicit cause, implicit context, and display suppression verified')

05: explicit cause, implicit context, and display suppression verified


**Why this works / trade-offs.** Use `from exc` when converting a low-level error into a domain error while retaining root-cause diagnostics. `from None` suppresses automatic display of context; the original may remain in `__context__`, so it does **not** erase sensitive values from exception objects.

### Problem 06 — Structured custom exception hierarchy

**Challenge.** Design a domain error base class and a validation subclass that record a stable `code`, `field`, and explanation. Expose a human-readable string but make programmatic callers branch on the *type or attributes*, not message parsing.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [12]:
class DomainError06(Exception):
    def __init__(self, code, message):
        self.code = code
        super().__init__(message)

class FieldError06(DomainError06):
    def __init__(self, field, message):
        self.field = field
        super().__init__('invalid_field', message)

def validated_age_06(value):
    if type(value) is not int or not 0 <= value <= 130:
        raise FieldError06('age', 'age must be an integer between 0 and 130')
    return value

**Examples and automated checks** — execute this cell to verify edge cases:

In [13]:
assert validated_age_06(35) == 35
for bad in [-1, 131, True, '45']:
    error = expect_error(FieldError06, validated_age_06, bad)
    assert error.code == 'invalid_field' and error.field == 'age'
    assert isinstance(error, DomainError06)
assert issubclass(FieldError06, Exception)
print('06: stable custom exception contract verified')

06: stable custom exception contract verified


**Why this works / trade-offs.** Inherit application errors from `Exception`, not directly from `BaseException`. Keep a narrow taxonomy and structured attributes; callers can catch `DomainError06` for an application boundary or `FieldError06` for more specialized recovery.

### Problem 07 — Exception object identity and arguments

**Challenge.** Distinguish *constructing* an exception from *raising* it. Store an exception instance, inspect `args`, `str`, and `repr`, then raise that same instance and prove identity is retained.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [14]:
def raise_existing_07(exc):
    raise exc

error_07 = ValueError('bad token', 17)

**Examples and automated checks** — execute this cell to verify edge cases:

In [15]:
assert error_07.args == ('bad token', 17)
assert 'bad token' in str(error_07)
assert 'ValueError' in repr(error_07)
assert error_07.__traceback__ is None  # not raised yet
caught_07 = expect_error(ValueError, raise_existing_07, error_07)
assert caught_07 is error_07
assert caught_07.__traceback__ is not None
print('07: constructing is not raising; identity verified')

07: constructing is not raising; identity verified


**Why this works / trade-offs.** `ValueError(...)` merely constructs an object. `raise` activates exception propagation. The default string representation of *multiple* exception args is tuple-like; avoid depending on exact string formatting as a machine-readable API.

### Problem 08 — A targeted multi-branch handler

**Challenge.** Write a parser for `"field:index"` that returns an item from a mapping of lists. Reject invalid syntax (`ValueError`), missing fields (`KeyError`), and out-of-range positions (`IndexError`) via distinct *public statuses*, without a broad exception handler.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [16]:
def lookup_status_08(mapping, expression):
    try:
        field, raw_index = expression.split(':')
        position = int(raw_index)
    except ValueError:
        return 'invalid-syntax', None
    try:
        item = mapping[field][position]
    except KeyError:
        return 'unknown-field', None
    except IndexError:
        return 'out-of-range', None
    else:
        return 'ok', item

**Examples and automated checks** — execute this cell to verify edge cases:

In [17]:
items_08 = {'a': [4, 5]}
assert lookup_status_08(items_08, 'a:1') == ('ok', 5)
assert lookup_status_08(items_08, 'a:x') == ('invalid-syntax', None)
assert lookup_status_08(items_08, 'a:2') == ('out-of-range', None)
assert lookup_status_08(items_08, 'z:0') == ('unknown-field', None)
assert lookup_status_08(items_08, 'a:0:1') == ('invalid-syntax', None)
print('08: each expected error receives its own recovery')

08: each expected error receives its own recovery


**Why this works / trade-offs.** The first `try` covers syntax/number conversion only; the second covers lookup only. Note: this exercise accepts negative indices in Python's normal way; add a separate `position < 0` validation if the API requires nonnegative positions.

---

## Part II — Narrow handlers, generators, and failure contracts

The input lesson demonstrates that catching every `Exception` inside a squaring generator accidentally hides a `TypeError`. The following problems generalize this lesson.

### Problem 09 — A generator that does not hide a data bug

**Challenge.** Implement `squares_upto(seq, limit)` that yields up to `limit` squares. It should stop if the sequence is exhausted, but propagate a `TypeError` if a member cannot be squared. Validate the limit before the generator is iterated, if practical.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [18]:
def squares_upto_09(seq, limit):
    if type(limit) is not int or limit < 0:
        raise ValueError('limit must be a nonnegative integer')
    def generate():
        for index in range(limit):
            try:
                item = seq[index]
            except IndexError:
                return
            yield item ** 2
    return generate()

**Examples and automated checks** — execute this cell to verify edge cases:

In [19]:
assert list(squares_upto_09([1, 2, 3], 50)) == [1, 4, 9]
assert list(squares_upto_09([1, 2, 3], 2)) == [1, 4]
assert list(squares_upto_09([], 10)) == []
expect_error(ValueError, squares_upto_09, [1], -1)
expect_error(TypeError, lambda: list(squares_upto_09([1, 'bad', 3], 9)))
print('09: normal exhaustion is distinct from invalid values')

09: normal exhaustion is distinct from invalid values


**Why this works / trade-offs.** The `try` surrounds `seq[index]` only. An `IndexError` raised by the exponentiation or unrelated downstream logic cannot be accidentally mistaken for collection exhaustion. Returning an inner generator lets the outer function validate arguments immediately.

### Problem 10 — Generator laziness and cleanup on close

**Challenge.** Create a generator that appends an `opened` event when iteration starts and a `closed` event when iteration ends or the caller closes it. Demonstrate that creating the generator does *not* start work, and that `close()` runs `finally` after starting.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [20]:
def managed_generator_10(events):
    events.append('opened')
    try:
        yield 1
        yield 2
    finally:
        events.append('closed')

**Examples and automated checks** — execute this cell to verify edge cases:

In [21]:
events_10 = []
gen_10 = managed_generator_10(events_10)
assert events_10 == []
assert next(gen_10) == 1 and events_10 == ['opened']
gen_10.close()
assert events_10 == ['opened', 'closed']
assert next(managed_generator_10([])) == 1
print('10: lazy initialization and deterministic close demonstrated')

10: lazy initialization and deterministic close demonstrated


**Why this works / trade-offs.** Generator bodies execute on iteration, not at creation. A started generator executes `finally` on explicit close, normal exhaustion, or exception. Prefer `with contextlib.closing(gen):` when a consumer may stop early and deterministic cleanup matters.

### Problem 11 — A reproducible check-then-act (LBYL) race

**Challenge.** Demonstrate why `if key in mapping: return mapping[key]` can fail if the mapping changes between the check and lookup. Create a deterministic mapping that removes a key during containment testing, then write an EAFP alternative using a narrow `KeyError` catch.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [22]:
class VanishingDict11(dict):
    def __contains__(self, key):
        present = super().__contains__(key)
        if present:
            self.pop(key)
        return present

def lbyl_get_11(mapping, key, default):
    if key in mapping:
        return mapping[key]
    return default

def eafp_get_11(mapping, key, default):
    try:
        return mapping[key]
    except KeyError:
        return default

**Examples and automated checks** — execute this cell to verify edge cases:

In [23]:
expect_error(KeyError, lbyl_get_11, VanishingDict11({'x': 1}), 'x', None)
assert eafp_get_11(VanishingDict11({'x': 1}), 'x', None) == 1
assert eafp_get_11({}, 'x', 'fallback') == 'fallback'
print('11: avoiding redundant presence checks prevents the simulated race')

11: avoiding redundant presence checks prevents the simulated race


**Why this works / trade-offs.** "Look before you leap" is not atomic. An EAFP lookup handles the operation's actual failure rather than relying on a stale check. Neither strategy alone solves general multithreaded transaction problems; appropriate synchronization may still be required.

### Problem 12 — Do not catch errors from consumer code

**Challenge.** Process each number through a user callback. Skip only conversion failures in the original input; if the callback raises even the same `ValueError`, allow it to propagate. Return `(results, skipped_count)`.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [24]:
def process_values_12(raw_values, callback):
    results, skipped = [], 0
    for raw in raw_values:
        try:
            number = int(raw)
        except (ValueError, TypeError):
            skipped += 1
            continue
        results.append(callback(number))  # intentionally outside the try
    return results, skipped

**Examples and automated checks** — execute this cell to verify edge cases:

In [25]:
assert process_values_12(['1', 'bad', '3'], lambda n: n * 2) == ([2, 6], 1)

def buggy_callback_12(n):
    if n == 2:
        raise ValueError('callback bug')
    return n

expect_error(ValueError, process_values_12, ['1', '2'], buggy_callback_12,
             contains='callback bug')
print('12: conversion errors skipped; callback failures remain visible')

12: conversion errors skipped; callback failures remain visible


**Why this works / trade-offs.** Keep the `try` block as small as possible. Although the callback may raise the same *type* as conversion, it is not an expected parse failure. Broad scopes accidentally convert unrelated bugs into "bad input".

### Problem 13 — Recover from a missing optional configuration value

**Challenge.** Read a required integer `workers` and optional integer `timeout` from a dictionary. Only the *optional missing key* should use a default. Missing required data, wrong types, and invalid integer syntax must remain distinguishable.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [26]:
def config_13(mapping):
    raw_workers = mapping['workers']
    try:
        raw_timeout = mapping['timeout']
    except KeyError:
        raw_timeout = '30'
    workers = int(raw_workers)
    timeout = int(raw_timeout)
    if workers < 1 or timeout < 0:
        raise ValueError('workers must be positive and timeout nonnegative')
    return {'workers': workers, 'timeout': timeout}

**Examples and automated checks** — execute this cell to verify edge cases:

In [27]:
assert config_13({'workers': '4'}) == {'workers': 4, 'timeout': 30}
assert config_13({'workers': 2, 'timeout': '0'})['timeout'] == 0
expect_error(KeyError, config_13, {'timeout': 10})
expect_error(ValueError, config_13, {'workers': 'not-a-number'})
expect_error(TypeError, config_13, {'workers': None})
print('13: only absence of optional config is defaulted')

13: only absence of optional config is defaulted


**Why this works / trade-offs.** Use defaults for *documented* optional fields, not for every failure. A failed conversion indicates present-but-invalid configuration and should be reported, not silently replaced.

---

## Part III — Files, cleanup, rollback, and exception safety

Resource management and consistency are advanced extensions of the lesson’s discussion of cleanup and database rollback.

### Problem 14 — Context-manager cleanup after a failure

**Challenge.** Implement `read_lines(stream)` using `contextlib.closing` so that a read failure still closes a custom stream. Use a controllable fake stream and verify cleanup on success and failure.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [28]:
class FakeStream14:
    def __init__(self, fail=False):
        self.fail = fail
        self.closed = False
    def read(self):
        if self.fail:
            raise OSError('device stopped')
        return 'one\ntwo\n'
    def close(self):
        self.closed = True

def read_lines_14(stream):
    with contextlib.closing(stream):
        return stream.read().splitlines()

**Examples and automated checks** — execute this cell to verify edge cases:

In [29]:
success_14 = FakeStream14()
assert read_lines_14(success_14) == ['one', 'two']
assert success_14.closed
failure_14 = FakeStream14(fail=True)
expect_error(OSError, read_lines_14, failure_14, contains='device stopped')
assert failure_14.closed
print('14: resources close on success and on exceptions')

14: resources close on success and on exceptions


**Why this works / trade-offs.** The `with` protocol guarantees the context manager's exit logic runs when a block exits. `closing` is suitable for objects with `close()` that lack a context-manager protocol; standard files already support `with open(...)`.

### Problem 15 — `ExitStack` unwinds partial acquisition

**Challenge.** Open a sequence of resources with a fake `open_fn` that may fail midway. Guarantee that already-acquired resources close in reverse order. Demonstrate successful and partially failed initialization.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [30]:
class Resource15:
    def __init__(self, name, events):
        self.name, self.events = name, events
    def close(self):
        self.events.append('close:' + self.name)

def acquire_resources_15(names, open_fn, events):
    with contextlib.ExitStack() as stack:
        resources = []
        for name in names:
            resource = open_fn(name)
            resources.append(resource)
            stack.callback(resource.close)
        events.append('work:' + ','.join(r.name for r in resources))
        return len(resources)

**Examples and automated checks** — execute this cell to verify edge cases:

In [31]:
events_15 = []
def open_good_15(name):
    events_15.append('open:' + name)
    return Resource15(name, events_15)
assert acquire_resources_15(['A', 'B'], open_good_15, events_15) == 2
assert events_15[-3:] == ['work:A,B', 'close:B', 'close:A']
events_15 = []
def open_bad_15(name):
    if name == 'C':
        raise OSError('cannot acquire C')
    return open_good_15(name)
expect_error(OSError, acquire_resources_15, ['A', 'B', 'C'], open_bad_15, events_15)
assert events_15 == ['open:A', 'open:B', 'close:B', 'close:A']
print('15: partially acquired resources are released LIFO')

15: partially acquired resources are released LIFO


**Why this works / trade-offs.** `ExitStack` handles a runtime-variable number of cleanup actions, including failures during acquisition. Register each callback immediately after acquisition. It runs them in last-in-first-out order, which is usually the correct dependency-unwinding order.

### Problem 16 — Transactional rollback without swallowing errors

**Challenge.** Create a context manager for a mutable ledger. Snapshot its entries at entry; keep writes on successful exit, but restore the snapshot and re-raise on **any escaping `BaseException`**. Test rollback on a normal exception and on a simulated `KeyboardInterrupt`.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [32]:
@contextlib.contextmanager
def ledger_transaction_16(ledger):
    snapshot = list(ledger)
    try:
        yield ledger
    except BaseException:
        ledger[:] = snapshot
        raise

**Examples and automated checks** — execute this cell to verify edge cases:

In [33]:
ledger_16 = [10]
with ledger_transaction_16(ledger_16) as tx:
    tx.append(20)
assert ledger_16 == [10, 20]

def fail_tx_16(error):
    with ledger_transaction_16(ledger_16) as tx:
        tx.extend([30, 40])
        raise error

original_16 = ValueError('rollback now')
assert expect_error(ValueError, fail_tx_16, original_16) is original_16
assert ledger_16 == [10, 20]
expect_error(KeyboardInterrupt, fail_tx_16, KeyboardInterrupt())
assert ledger_16 == [10, 20]
print('16: commit, rollback, and termination propagation verified')

16: commit, rollback, and termination propagation verified


**Why this works / trade-offs.** This is a rare *valid* `except BaseException`: mandatory cleanup runs, then a bare `raise` propagates **everything**, including termination signals. The list snapshot is intentionally simple and shallow; a real DB uses the database's transactional rollback, and deep mutable state requires a stronger snapshot strategy.

### Problem 17 — Atomic file replacement with failure injection

**Challenge.** Write UTF-8 text using a temporary file in the destination directory and `os.replace`. Accept an injectable `writer` that may fail before replacement. Check that an existing target remains unchanged after failure and that no temporary files leak. Use only a temporary directory in the test.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [34]:
def write_text_atomically_17(destination, text, writer=None):
    destination = Path(destination)
    writer = writer or (lambda fh, value: fh.write(value))
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode='w', encoding='utf-8', newline='',
            dir=destination.parent, prefix='.staging-', delete=False
        ) as fh:
            temp_path = Path(fh.name)
            writer(fh, text)
            fh.flush()
            os.fsync(fh.fileno())
        os.replace(temp_path, destination)
        temp_path = None
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)

**Examples and automated checks** — execute this cell to verify edge cases:

In [35]:
with tempfile.TemporaryDirectory() as directory_17:
    target_17 = Path(directory_17) / 'data.txt'
    target_17.write_text('old', encoding='utf-8')
    def broken_writer_17(fh, text):
        fh.write('partial')
        raise OSError('disk simulator')
    expect_error(OSError, write_text_atomically_17, target_17, 'new', broken_writer_17)
    assert target_17.read_text(encoding='utf-8') == 'old'
    assert sorted(p.name for p in Path(directory_17).iterdir()) == ['data.txt']
    write_text_atomically_17(target_17, 'new')
    assert target_17.read_text(encoding='utf-8') == 'new'
print('17: failure leaves target unchanged and temp file removed')

17: failure leaves target unchanged and temp file removed


**Why this works / trade-offs.** Use an on-filesystem temporary file and replacement to avoid truncated target files on write failure. Atomic replacement depends on filesystem behavior; `fsync` of the file is not the same as a full crash-durability guarantee (which may also require directory syncing and platform-specific handling).

### Problem 18 — Cleanup can fail too: preserve both failures

**Challenge.** An operation raises `ValueError`; its `close()` also raises `OSError`. Write a wrapper that retains *both* failures in `ExceptionGroup` rather than losing the operation error under the cleanup error. Demonstrate the success-path cleanup-only failure too.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [36]:
class FaultyResource18:
    def close(self):
        raise OSError('cleanup failed')

def run_with_dual_errors_18(operation, resource):
    primary = None
    try:
        return operation()
    except Exception as exc:
        primary = exc
        raise
    finally:
        try:
            resource.close()
        except Exception as cleanup_exc:
            if primary is not None:
                raise ExceptionGroup('operation and cleanup failed',
                                     [primary, cleanup_exc]) from primary
            raise

**Examples and automated checks** — execute this cell to verify edge cases:

In [37]:
def faulty_operation_18():
    raise ValueError('operation failed')

group_18 = expect_error(ExceptionGroup, run_with_dual_errors_18,
                        faulty_operation_18, FaultyResource18())
assert [type(e) for e in group_18.exceptions] == [ValueError, OSError]
assert isinstance(group_18.__cause__, ValueError)
expect_error(OSError, run_with_dual_errors_18, lambda: 99, FaultyResource18())
print('18: operation and cleanup failures remain inspectable')

18: operation and cleanup failures remain inspectable


**Why this works / trade-offs.** A `finally` exception can replace the active exception as the externally raised one. `ExceptionGroup` is a Python 3.11+ mechanism for surfacing several ordinary exceptions together. In production, a well-designed resource's context manager or a library's documented cleanup behavior should usually take precedence over rolling your own.

---

## Part IV — Translation, observability, and external boundaries

Preserve diagnostics for developers while making boundaries predictable for callers.

### Problem 19 — Translate a storage exception at an API boundary

**Challenge.** Build a repository API `get_user` that converts only an anticipated `KeyError` into `UserNotFound`, retaining the original cause and the missing identifier. Unexpected `TypeError` from a broken storage layer must propagate unaltered.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [38]:
class UserNotFound19(Exception):
    def __init__(self, user_id):
        self.user_id = user_id
        super().__init__(f'user not found: {user_id}')

def get_user_19(store, user_id):
    try:
        return store[user_id]
    except KeyError as exc:
        raise UserNotFound19(user_id) from exc

**Examples and automated checks** — execute this cell to verify edge cases:

In [39]:
assert get_user_19({'u1': {'name': 'Ada'}}, 'u1')['name'] == 'Ada'
missing_19 = expect_error(UserNotFound19, get_user_19, {}, 'u2')
assert missing_19.user_id == 'u2' and isinstance(missing_19.__cause__, KeyError)
class BrokenStore19:
    def __getitem__(self, key):
        raise TypeError('storage bug')
expect_error(TypeError, get_user_19, BrokenStore19(), 'u2')
print('19: domain translation does not disguise unrelated storage defects')

19: domain translation does not disguise unrelated storage defects


**Why this works / trade-offs.** Translate errors at module boundaries where the abstraction changes: the caller should not need to know a repository uses a dictionary. Retain the original error through explicit chaining and allow unexpected errors to surface for investigation.

### Problem 20 — Log a traceback once and keep it propagating

**Challenge.** Log an anticipated `ValueError` with its traceback and re-raise it without changing identity. Capture the log in memory for a test; avoid logging raw secrets. Show where logging belongs (at a responsible boundary, not every nested function).

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [40]:
def transform_and_log_20(text, logger):
    try:
        return int(text)
    except ValueError:
        logger.exception('Could not parse input at the job boundary')
        raise

log_output_20 = io.StringIO()
logger_20 = logging.getLogger('notebook.exceptions.20')
logger_20.handlers.clear()
logger_20.propagate = False
logger_20.setLevel(logging.ERROR)
handler_20 = logging.StreamHandler(log_output_20)
logger_20.addHandler(handler_20)

**Examples and automated checks** — execute this cell to verify edge cases:

In [41]:
assert transform_and_log_20('42', logger_20) == 42
log_output_20.seek(0)
log_output_20.truncate(0)
expect_error(ValueError, transform_and_log_20, 'bad', logger_20)
logged_20 = log_output_20.getvalue()
assert 'Could not parse input' in logged_20
assert 'Traceback' in logged_20 and 'ValueError' in logged_20
logger_20.removeHandler(handler_20)
handler_20.close()
print('20: traceback logged once; exception still propagates')

20: traceback logged once; exception still propagates


**Why this works / trade-offs.** `logger.exception(...)` includes exception information inside an active handler; do not catch and log at every layer, or the same failure will create repetitive logs. This demo's sample data is harmless; in real applications avoid secrets and personally identifying data in log messages and traces.

### Problem 21 — Validation: JSON syntax vs schema errors

**Challenge.** Create a strict input function for JSON objects of shape `{"id": positive_int, "name": nonempty_string}`. Distinguish syntax failure (`JsonSyntaxError`) from schema failure (`JsonSchemaError`) and retain parser exceptions as causes. Reject booleans as IDs.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [42]:
class JsonInputError21(Exception):
    pass
class JsonSyntaxError21(JsonInputError21):
    pass
class JsonSchemaError21(JsonInputError21):
    pass

def decode_person_21(payload):
    try:
        data = json.loads(payload)
    except json.JSONDecodeError as exc:
        raise JsonSyntaxError21('malformed JSON') from exc
    if not isinstance(data, dict):
        raise JsonSchemaError21('expected JSON object')
    if set(data) != {'id', 'name'}:
        raise JsonSchemaError21('expected exactly id and name')
    if type(data['id']) is not int or data['id'] <= 0:
        raise JsonSchemaError21('id must be a positive integer')
    if not isinstance(data['name'], str) or not data['name'].strip():
        raise JsonSchemaError21('name must be nonempty text')
    return {'id': data['id'], 'name': data['name'].strip()}

**Examples and automated checks** — execute this cell to verify edge cases:

In [43]:
assert decode_person_21('{"id": 2, "name": " Ada "}') == {'id': 2, 'name': 'Ada'}
syntax_21 = expect_error(JsonSyntaxError21, decode_person_21, '{broken')
assert isinstance(syntax_21.__cause__, json.JSONDecodeError)
for bad_21 in ['[]', '{}', '{"id": true, "name": "A"}',
               '{"id": 1, "name": "  "}', '{"id": 1, "name": "A", "x": 9}']:
    expect_error(JsonSchemaError21, decode_person_21, bad_21)
print('21: syntax and schema errors remain distinguishable')

21: syntax and schema errors remain distinguishable


**Why this works / trade-offs.** `json.loads` validates syntax, not your application schema. Translate parser errors precisely and validate types and fields separately. A boolean is an `int` subclass in Python, so `type(value) is int` is intentional when `true` must be disallowed.

### Problem 22 — No exception-driven currency guessing

**Challenge.** Parse decimal money strings safely. Reject non-text inputs, invalid decimals, `NaN`, infinity, and values with more than two decimal places. Convert expected decimal parsing errors into a `MoneyError`, keeping the original cause; use exact Decimal arithmetic.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [44]:
class MoneyError22(ValueError):
    pass

def parse_money_22(raw):
    if not isinstance(raw, str):
        raise MoneyError22('amount must be text')
    try:
        amount = Decimal(raw)
    except InvalidOperation as exc:
        raise MoneyError22('invalid decimal amount') from exc
    if not amount.is_finite() or amount.as_tuple().exponent < -2:
        raise MoneyError22('amount needs at most two decimal places and must be finite')
    if amount < 0:
        raise MoneyError22('amount must be nonnegative')
    return amount.quantize(Decimal('0.01'))

**Examples and automated checks** — execute this cell to verify edge cases:

In [45]:
assert parse_money_22('12.30') == Decimal('12.30')
assert parse_money_22('0') == Decimal('0.00')
for bad_22 in ['nope', 'NaN', 'Infinity', '-1.00', '1.001', None]:
    expect_error(MoneyError22, parse_money_22, bad_22)
assert isinstance(expect_error(MoneyError22, parse_money_22,
                               'bad').__cause__, InvalidOperation)
print('22: exact money validation and cause chaining verified')

22: exact money validation and cause chaining verified


**Why this works / trade-offs.** Decimal avoids binary floating-point surprises for monetary values. The format contract in this exercise rejects excess decimal places instead of silently rounding; a different application may explicitly choose a rounding policy. Avoid a catch-all that replaces arbitrary errors with zero.

### Problem 23 — Per-record batch errors without masking bugs

**Challenge.** Given rows of `(user_id, raw_age)`, return good records and structured errors for invalid ages. Treat conversion `ValueError` and `TypeError` as expected bad records but do not catch unrelated failures. Retain each input index and field.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [46]:
@dataclass(frozen=True)
class RecordIssue23:
    position: int
    field: str
    code: str

def validate_batch_23(rows):
    good, issues = [], []
    for index, (user_id, raw_age) in enumerate(rows):
        try:
            age = int(raw_age)
        except (ValueError, TypeError):
            issues.append(RecordIssue23(index, 'age', 'not_integer'))
            continue
        if isinstance(raw_age, bool) or not 0 <= age <= 130:
            issues.append(RecordIssue23(index, 'age', 'out_of_range'))
            continue
        good.append({'id': user_id, 'age': age})
    return good, issues

**Examples and automated checks** — execute this cell to verify edge cases:

In [47]:
valid_23, issues_23 = validate_batch_23([
    ('a', '21'), ('b', '?'), ('c', -1), ('d', 30), ('e', True)
])
assert valid_23 == [{'id': 'a', 'age': 21}, {'id': 'd', 'age': 30}]
assert [(i.position, i.code) for i in issues_23] == [
    (1, 'not_integer'), (2, 'out_of_range'), (4, 'out_of_range')
]
# Programmer error in row structure is not silently turned into a bad age.
expect_error(ValueError, validate_batch_23, [('incomplete',)])
print('23: explicit partial success and structured record issues verified')

23: explicit partial success and structured record issues verified


**Why this works / trade-offs.** A batch may deliberately tolerate *documented* input errors while continuing. Returning two explicit collections lets the caller choose whether partial success is acceptable. This simplified parser accepts `1.5` by `int(1.5)` truncation; a strict real-world schema should additionally constrain incoming age types before conversion.

### Problem 24 — Optional imports: distinguish missing plugin from broken dependencies

**Challenge.** Implement `load_optional(module_name, loader=...)`. Return `None` only when *the requested module itself* is missing. A `ModuleNotFoundError` for a different dependency from inside that module must propagate. Use injection rather than depending on installed packages.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [48]:
def load_optional_24(module_name, loader=importlib.import_module):
    try:
        return loader(module_name)
    except ModuleNotFoundError as exc:
        if exc.name == module_name:
            return None
        raise

**Examples and automated checks** — execute this cell to verify edge cases:

In [49]:
class DummyModule24:
    name = 'loaded'

def loader_24(name):
    if name == 'available':
        return DummyModule24()
    if name == 'missing':
        raise ModuleNotFoundError("No module named 'missing'", name='missing')
    raise ModuleNotFoundError("No module named 'internal_dep'", name='internal_dep')

assert load_optional_24('available', loader_24).name == 'loaded'
assert load_optional_24('missing', loader_24) is None
expect_error(ModuleNotFoundError, load_optional_24, 'broken', loader_24)
print('24: an internal import failure cannot masquerade as an absent plugin')

24: an internal import failure cannot masquerade as an absent plugin


**Why this works / trade-offs.** Optional dependencies are often implemented incorrectly with `except ImportError: pass`: that hides missing sub-dependencies and genuine module import defects. Inspect `ModuleNotFoundError.name`. For dotted module requests and import hooks, design an explicit policy for missing parents and side-effect failures.

---

## Part V — Retrying, concurrency, and grouped failures

Retry only transient, repeatable operations; cancellation and grouped errors deserve their own exception contracts.

### Problem 25 — Bounded retry with injectable delay

**Challenge.** Write `retry_transient(fn, attempts, pause)` that retries **only `ConnectionError`**, validates `attempts`, invokes an injected (fake) pause between failed attempts, and raises the original final exception after exhausting retries. Do not retry programming bugs.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [50]:
def retry_transient_25(fn, attempts=3, pause=lambda attempt: None):
    if type(attempts) is not int or attempts < 1:
        raise ValueError('attempts must be a positive integer')
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except ConnectionError:
            if attempt == attempts:
                raise
            pause(attempt)
    raise AssertionError('unreachable')

**Examples and automated checks** — execute this cell to verify edge cases:

In [51]:
remaining_25 = deque([ConnectionError('temporary'),
                      ConnectionError('temporary'), 'OK'])
pauses_25 = []
def flaky_25():
    result = remaining_25.popleft()
    if isinstance(result, Exception):
        raise result
    return result
assert retry_transient_25(flaky_25, 3, pauses_25.append) == 'OK'
assert pauses_25 == [1, 2]
last_error_25 = ConnectionError('down')
assert expect_error(ConnectionError, retry_transient_25,
                    lambda: (_ for _ in ()).throw(last_error_25), 2) is last_error_25
expect_error(ValueError, retry_transient_25, lambda: None, 0)
expect_error(TypeError, retry_transient_25,
             lambda: (_ for _ in ()).throw(TypeError('bug')), 3)
print('25: bounded, targeted, fake-time retry verified')

25: bounded, targeted, fake-time retry verified


**Why this works / trade-offs.** Retries must be bounded, selective and safe for the operation: a repeated payment or write may duplicate a side effect without an idempotency key. A production retry policy may add capped exponential backoff, jitter, timeouts, metrics, and an explicit error budget. This example never sleeps.

### Problem 26 — A deterministic circuit breaker

**Challenge.** Implement a circuit breaker with CLOSED and OPEN states. After `failure_threshold` consecutive `ConnectionError`s, reject calls with a separate `CircuitOpen`. Use an injected clock, allow one trial after `reset_after`, and reset after success. Non-transient bugs must propagate.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [52]:
class CircuitOpen26(Exception):
    pass

class CircuitBreaker26:
    def __init__(self, failure_threshold, reset_after, clock):
        if type(failure_threshold) is not int or failure_threshold < 1 or reset_after < 0:
            raise ValueError('invalid circuit settings')
        self.failure_threshold = failure_threshold
        self.reset_after = reset_after
        self.clock = clock
        self.failures = 0
        self.opened_at = None

    def call(self, fn):
        if self.opened_at is not None:
            if self.clock() - self.opened_at < self.reset_after:
                raise CircuitOpen26('temporarily unavailable')
            # Simplified single-threaded half-open trial.
            self.opened_at = None
            self.failures = 0
        try:
            result = fn()
        except ConnectionError:
            self.failures += 1
            if self.failures >= self.failure_threshold:
                self.opened_at = self.clock()
            raise
        else:
            self.failures = 0
            return result

**Examples and automated checks** — execute this cell to verify edge cases:

In [53]:
clock_26 = [0.0]
breaker_26 = CircuitBreaker26(2, 5, lambda: clock_26[0])
def unavailable_26():
    raise ConnectionError('network')
expect_error(ConnectionError, breaker_26.call, unavailable_26)
expect_error(ConnectionError, breaker_26.call, unavailable_26)
expect_error(CircuitOpen26, breaker_26.call, lambda: 'blocked')
clock_26[0] = 5.0
assert breaker_26.call(lambda: 'recovered') == 'recovered'
assert breaker_26.failures == 0 and breaker_26.opened_at is None
expect_error(TypeError, breaker_26.call,
             lambda: (_ for _ in ()).throw(TypeError('bug')))
assert breaker_26.failures == 0
print('26: open, short-circuit, timed trial, and recovery verified')

26: open, short-circuit, timed trial, and recovery verified


**Why this works / trade-offs.** This instructional design is explicitly **single-threaded**. A concurrent breaker requires synchronization and controlled half-open probes. A circuit breaker stops repeatedly issuing calls when a dependency is failing; it must not mistake arbitrary code bugs for connectivity problems.

### Problem 27 — Selective handling of an `ExceptionGroup`

**Challenge.** Raise a group containing `ValueError`, `TypeError`, and `OSError`. Use `except*` to collect expected validation errors (`ValueError`, `TypeError`) and let `OSError` propagate as a residual group. Assert both subgroup contents and the residual exception type.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [54]:
def grouped_failure_27():
    raise ExceptionGroup('batch', [
        ValueError('bad field'), TypeError('wrong type'), OSError('disk down')
    ])

def handle_group_27():
    handled = []
    residual = []
    try:
        try:
            grouped_failure_27()
        except* (ValueError, TypeError) as group:
            handled.extend(type(exc).__name__ for exc in group.exceptions)
    except ExceptionGroup as group:
        residual.extend(type(exc).__name__ for exc in group.exceptions)
    return handled, residual

**Examples and automated checks** — execute this cell to verify edge cases:

In [55]:
handled_27, residual_27 = handle_group_27()
assert handled_27 == ['ValueError', 'TypeError']
assert residual_27 == ['OSError']
assert isinstance(expect_error(ExceptionGroup, grouped_failure_27), ExceptionGroup)
print('27: except* handled matches and preserved the remaining error')

27: except* handled matches and preserved the remaining error


**Why this works / trade-offs.** `except*` partitions exception groups by type. It does not silently erase unmatched exceptions, which propagate as a remainder. `except` and `except*` cannot appear in the same `try` statement; nesting above is intentional.

### Problem 28 — Async batch: results and errors without disguising exceptions

**Challenge.** Use `asyncio.gather(..., return_exceptions=True)` with three coroutines: successful result, `ValueError`, and `ConnectionError`. Return results and structured failures, then show that all exceptions are still visible and typed. Drive the coroutine with `asyncio.run` only outside Jupyter? **In Jupyter, use top-level `await`.**

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [56]:
async def task_28(value):
    await asyncio.sleep(0)  # yields once; no real wait
    if value == 'invalid':
        raise ValueError('invalid input')
    if value == 'offline':
        raise ConnectionError('offline')
    return value.upper()

async def classify_async_28(values):
    completed = await asyncio.gather(
        *(task_28(value) for value in values), return_exceptions=True
    )
    successful, failures = [], []
    for index, result in enumerate(completed):
        if isinstance(result, BaseException):
            failures.append((index, type(result).__name__, result))
        else:
            successful.append((index, result))
    return successful, failures

**Examples and automated checks** — execute this cell to verify edge cases:

In [57]:
successes_28, failures_28 = await classify_async_28(['ok', 'invalid', 'offline'])
assert successes_28 == [(0, 'OK')]
assert [(index, kind) for index, kind, _ in failures_28] == [
    (1, 'ValueError'), (2, 'ConnectionError')
]
assert isinstance(failures_28[0][2], ValueError)
print('28: all async task outcomes were examined, not discarded')

28: all async task outcomes were examined, not discarded


**Why this works / trade-offs.** `gather(return_exceptions=True)` returns exceptions as values; every caller must explicitly classify them. In Jupyter, top-level `await` uses the running event loop; calling `asyncio.run()` from that same loop would fail. For structured concurrency, Python 3.11 `TaskGroup` groups non-cancellation failures and cancels sibling tasks.

### Problem 29 — Cooperative cancellation is not an ordinary error

**Challenge.** Create a coroutine that records cleanup when it is cancelled. Start it, yield control so it begins, cancel it, and verify that `CancelledError` propagates and `finally` executes. Avoid `except BaseException: return fallback`.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [58]:
async def cancellable_worker_29(events):
    events.append('started')
    try:
        await asyncio.Event().wait()
    finally:
        events.append('cleaned')

async def cancellation_demo_29():
    events = []
    worker = asyncio.create_task(cancellable_worker_29(events))
    await asyncio.sleep(0)
    worker.cancel()
    try:
        await worker
    except asyncio.CancelledError:
        events.append('cancelled-observed')
    return events

**Examples and automated checks** — execute this cell to verify edge cases:

In [59]:
events_29 = await cancellation_demo_29()
assert events_29 == ['started', 'cleaned', 'cancelled-observed']
assert issubclass(asyncio.CancelledError, BaseException)
print('29: cancellation propagated to the caller, cleanup ran')

29: cancellation propagated to the caller, cleanup ran


**Why this works / trade-offs.** In modern Python, `asyncio.CancelledError` inherits from `BaseException`, so ordinary `except Exception` does not intercept it. A coroutine may catch cancellation to clean up, but should normally re-raise it rather than make cancellation disappear. `finally` is often simpler.

### Problem 30 — Warnings are not exceptions by default

**Challenge.** Emit a `DeprecationWarning` from a helper. Capture it without polluting the notebook's global warning filters, and demonstrate converting it to an exception locally for a strict compatibility test.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [60]:
def legacy_api_30():
    warnings.warn('use new_api instead', DeprecationWarning, stacklevel=2)
    return 42

def capture_warning_30():
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always', DeprecationWarning)
        result = legacy_api_30()
    return result, recorded

def warning_as_error_30():
    with warnings.catch_warnings():
        warnings.simplefilter('error', DeprecationWarning)
        return legacy_api_30()

**Examples and automated checks** — execute this cell to verify edge cases:

In [61]:
result_30, warnings_30 = capture_warning_30()
assert result_30 == 42 and len(warnings_30) == 1
assert issubclass(warnings_30[0].category, DeprecationWarning)
expect_error(DeprecationWarning, warning_as_error_30)
print('30: warning capture and warning-as-error verified')

30: warning capture and warning-as-error verified


**Why this works / trade-offs.** Warnings signal nonfatal concerns under default filtering. `warnings.catch_warnings()` provides a localized scope for testing; do not permanently change global warning settings just to demonstrate one behavior.

---

## Part VI — Applied integration, anti-pattern diagnosis, and robust tests

The final exercises combine earlier exception contracts into end-to-end workflows and explicit tests.

### Problem 31 — Diagnose and repair three anti-patterns

**Challenge.** Repair code that (1) silently returns `None` for every error, (2) discards the original traceback with an unrelated translation, and (3) uses `finally: return` (which masks exceptions). Implement a strict integer division function that translates only invalid numeric inputs, chains the cause, preserves `ZeroDivisionError`, and always records cleanup without returning from `finally`.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [62]:
class NumericInputError31(ValueError):
    pass

def robust_division_31(left, right, events):
    try:
        try:
            a, b = int(left), int(right)
        except (ValueError, TypeError) as exc:
            raise NumericInputError31('arguments must be integers') from exc
        return a / b
    finally:
        events.append('cleanup')

**Examples and automated checks** — execute this cell to verify edge cases:

In [63]:
events_31 = []
assert robust_division_31('8', '2', events_31) == 4.0
assert events_31 == ['cleanup']
events_31.clear()
bad_31 = expect_error(NumericInputError31, robust_division_31,
                      'oops', 2, events_31)
assert isinstance(bad_31.__cause__, ValueError)
assert events_31 == ['cleanup']
events_31.clear()
expect_error(ZeroDivisionError, robust_division_31, 8, 0, events_31)
assert events_31 == ['cleanup']
print('31: only documented errors translated; finally never masks return/error')

31: only documented errors translated; finally never masks return/error


**Why this works / trade-offs.** Never `return` from `finally`: it overrides a preceding return or active exception. Translating precise expected parse failures with `raise ... from exc` keeps root-cause information. A division-by-zero is not an integer-parsing failure, so leave it visible.

### Problem 32 — CAPSTONE: all-or-nothing batch import

**Challenge.** Build an all-or-nothing importer for rows with `id` and `amount`. Use the JSON schema conventions from earlier examples, money validation, and an in-memory ledger transaction. Duplicate IDs and invalid rows must raise a dedicated `ImportError32` with the **row number** and original cause. The ledger must be unchanged if any row fails; on success it commits all rows.

Try writing your own implementation before revealing the solution by scrolling.

**Worked solution** — a complete, runnable reference implementation:

In [64]:
class ImportError32(Exception):
    def __init__(self, row, message):
        self.row = row
        super().__init__(f'row {row}: {message}')

def normalize_row_32(row):
    if not isinstance(row, dict):
        raise ValueError('each row must be a dictionary')
    if set(row) != {'id', 'amount'}:
        raise ValueError('row must have exactly id and amount')
    if not isinstance(row['id'], str) or not row['id'].strip():
        raise ValueError('id must be nonempty text')
    return row['id'].strip(), parse_money_22(row['amount'])

def import_batch_32(ledger, rows):
    # ledger is a mutable list of (id, Decimal) tuples.
    with ledger_transaction_16(ledger):
        ids = {entry_id for entry_id, _ in ledger}
        for number, row in enumerate(rows, start=1):
            try:
                entry_id, amount = normalize_row_32(row)
                if entry_id in ids:
                    raise ValueError('duplicate id')
            except (ValueError, TypeError) as exc:
                raise ImportError32(number, 'invalid or duplicate record') from exc
            ledger.append((entry_id, amount))
            ids.add(entry_id)
    return len(ledger)

**Examples and automated checks** — execute this cell to verify edge cases:

In [65]:
ledger_32 = [('old', Decimal('1.00'))]
assert import_batch_32(ledger_32, [
    {'id': 'a', 'amount': '2.25'}, {'id': 'b', 'amount': '3'}
]) == 3
assert ledger_32[-2:] == [('a', Decimal('2.25')), ('b', Decimal('3.00'))]
before_32 = list(ledger_32)
bad_32 = expect_error(ImportError32, import_batch_32, ledger_32, [
    {'id': 'c', 'amount': '4.00'}, {'id': 'c', 'amount': '5.00'}
])
assert bad_32.row == 2 and isinstance(bad_32.__cause__, ValueError)
assert ledger_32 == before_32
bad_amount_32 = expect_error(ImportError32, import_batch_32, ledger_32, [
    {'id': 'd', 'amount': 'nan?'}
])
assert bad_amount_32.row == 1 and isinstance(bad_amount_32.__cause__, MoneyError22)
assert ledger_32 == before_32
print('32: capstone committed valid batch and rolled back both failures')

32: capstone committed valid batch and rolled back both failures


**Why this works / trade-offs.** The importer applies validation *before appending each record*, but the transaction provides the stronger all-or-nothing property for the batch. Duplicate detection covers both existing and newly imported rows. The caller receives a stable row number, and `__cause__` retains the concrete validation failure. This in-memory snapshot does not provide cross-process ACID guarantees.

---

## Final assessment — independent regression suite

These are ordinary `unittest` checks on the worked solutions. The earlier cells use compact `assert`s for readability; a real project should use `unittest`/`pytest` and run under continuous integration. Complete the notebook top to bottom before this final suite.

In [66]:
class ExceptionWorkbookRegressionTests(unittest.TestCase):
    def test_hierarchy_precedence(self):
        self.assertIs(first_matching_handler(IndexError(), [LookupError, IndexError]),
                      LookupError)

    def test_index_sentinel(self):
        self.assertIsNone(get_at([None], 0, default='missing'))
        with self.assertRaises(IndexError):
            get_at([], 0)

    def test_generator_propagates_typeerror(self):
        with self.assertRaises(TypeError):
            list(squares_upto_09([3, object()], 5))

    def test_transaction_rolls_back(self):
        items = [1, 2]
        with self.assertRaisesRegex(ValueError, 'abort'):
            with ledger_transaction_16(items):
                items.append(99)
                raise ValueError('abort')
        self.assertEqual(items, [1, 2])

    def test_atomic_writer_rolls_back(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / 'value.txt'
            path.write_text('BEFORE', encoding='utf-8')
            def fail(fh, text):
                fh.write('HALF')
                raise OSError('write failure')
            with self.assertRaises(OSError):
                write_text_atomically_17(path, 'AFTER', fail)
            self.assertEqual(path.read_text(encoding='utf-8'), 'BEFORE')
            self.assertEqual(len(list(Path(directory).iterdir())), 1)

    def test_exception_cause(self):
        with self.assertRaises(UserNotFound19) as caught:
            get_user_19({}, 'nobody')
        self.assertIsInstance(caught.exception.__cause__, KeyError)

    def test_json_validation(self):
        with self.assertRaises(JsonSchemaError21):
            decode_person_21('{"id": false, "name": "Ada"}')
        self.assertEqual(decode_person_21('{"id": 1, "name": "Ada"}')['id'], 1)

    def test_retry_has_finite_budget(self):
        invocations = []
        def fail():
            invocations.append(1)
            raise ConnectionError('unavailable')
        with self.assertRaises(ConnectionError):
            retry_transient_25(fail, attempts=4)
        self.assertEqual(len(invocations), 4)

    def test_group_preserves_residual(self):
        self.assertEqual(handle_group_27()[1], ['OSError'])

    def test_import_rolls_back_everything(self):
        ledger = [('seed', Decimal('3.00'))]
        with self.assertRaises(ImportError32) as caught:
            import_batch_32(ledger, [
                {'id': 'new', 'amount': '2.00'},
                {'id': 'new', 'amount': '3.00'}
            ])
        self.assertEqual(caught.exception.row, 2)
        self.assertEqual(ledger, [('seed', Decimal('3.00'))])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ExceptionWorkbookRegressionTests)
stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
print(stream.getvalue())
assert result.wasSuccessful(), 'The full workbook regression suite failed.'
print(f'WORKBOOK VERIFIED: {result.testsRun} regression tests passed.')

test_atomic_writer_rolls_back (__main__.ExceptionWorkbookRegressionTests.test_atomic_writer_rolls_back) ... ok
test_exception_cause (__main__.ExceptionWorkbookRegressionTests.test_exception_cause) ... ok
test_generator_propagates_typeerror (__main__.ExceptionWorkbookRegressionTests.test_generator_propagates_typeerror) ... ok
test_group_preserves_residual (__main__.ExceptionWorkbookRegressionTests.test_group_preserves_residual) ... ok
test_hierarchy_precedence (__main__.ExceptionWorkbookRegressionTests.test_hierarchy_precedence) ... ok
test_import_rolls_back_everything (__main__.ExceptionWorkbookRegressionTests.test_import_rolls_back_everything) ... ok
test_index_sentinel (__main__.ExceptionWorkbookRegressionTests.test_index_sentinel) ... ok
test_json_validation (__main__.ExceptionWorkbookRegressionTests.test_json_validation) ... ok
test_retry_has_finite_budget (__main__.ExceptionWorkbookRegressionTests.test_retry_has_finite_budget) ... ok
test_transaction_rolls_back (__main__.Exception

---

## Takeaway: a practical exception-handling checklist

1. **Define the contract:** which inputs and failures should callers expect?
2. **Keep `try` small:** only surround the statement that can produce the anticipated error.
3. **Catch narrowly:** choose the actual recoverable class; put subclasses first.
4. **Do not obscure bugs:** never turn arbitrary exceptions into empty results, success, or "invalid input".
5. **Preserve causal information:** bare `raise` for rethrow; `raise DomainError(...) from exc` for translation.
6. **Clean up deterministically:** use `with`, `ExitStack`, and `finally`; avoid `return` inside `finally`.
7. **Consider exception safety:** on failure, does the operation leave state unchanged, partially modified, or rolled back?
8. **Retry only carefully:** bounded attempts, known transient errors, idempotent operations, and testable time sources.
9. **Respect async cancellation:** run cleanup, but let cancellation normally propagate.
10. **Test the failure path:** prove error types, causes, exception identity where relevant, resource cleanup, rollback, and absence of unwanted side effects.

### Further practice (solutions are already illustrated above)

- Modify Problem 09 so it rejects negative element values, and test that this new validation is never mistaken for exhaustion.
- Extend Problem 16 to snapshot a nested structure without aliasing; explain the memory/consistency tradeoff.
- Extend Problem 17 with a directory-syncing policy on systems where durable replacement is required.
- Extend Problem 25 to use an injectable exponential backoff function and test the exact delays without sleeping.
- Extend Problem 26 for concurrent calls, controlling the single permitted half-open probe with a lock.
- Extend Problem 32 with an explicit `dry_run` API that validates every row without changing the ledger.

**Provenance:** The provided notebook supplied the foundational concepts and the unsafe broad-handler generator scenario; the 32 numbered problems, implementations, edge-case tests, and production-oriented extensions here are new instructional material. Historical traceback screenshots and the original Python 3.6.7 metadata were intentionally not copied into executable cells; this workbook targets Python 3.11+ to teach modern exception groups and notes.